In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")
#/kaggle/input/q1-ka-ai-2026/Q1_data.csv
print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
df_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(df_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
#target distribution (delivery_time)
# ماعرفت اضبط التوقيت بالقراف بحيث يكون مضبوط
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=12, edgecolor='black')
plt.title('Delivery Time')
plt.xlabel('Delivery Time')
plt.ylabel('Time')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean = df.copy()
df_clean.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
#Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
print("Missing values:")
print(df_clean.isnull().sum())
#عبيت القيم الفاضية للDelivery_Time بالمتوسط الحسابي
df_clean['Delivery_Time']= df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].median())
#عبيت القيم الفاضية Courier_Experience_yrs بالمتوسط الحسابي
df_clean['Courier_Experience_yrs']=df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].median())
#عبيت القيم الفاضية Time_of_Day  بعبارة unknown
df_clean['Time_of_Day']=df_clean['Time_of_Day'] = df_clean['Time_of_Day'].fillna('unknown')
df_clean['Traffic_Level']=df_clean['Traffic_Level'] = df_clean['Traffic_Level'].fillna('Medium')

df_clean['Weather']=df_clean['Weather'] = df_clean['Weather'].fillna('Clear')



In [ ]:
print("Missing values:")
print(df_clean.isnull().sum())

In [ ]:
##Check and remove duplicates
def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
#Encode categorical variables if needed (Bonus if used One Hot Encoding)
# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 4: Write your code here:
#Apply feature scaling for all features (Use StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_clean, "Delivery_Time")

In [ ]:
Split the dataset into features (X) and target (y)
#Apply feature scaling for all features (Use StandardScaler)
feature_cols = ['Order_ID','Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# Task 2,3,4,5: Write your code here:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
    model.fit(X_train_scaled, y_train)
    print("Model trained!")
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: